# 03 - Numeric tables as LaTeX

Writes `work/tex/tables_auto.tex` and `work/tex/tables_extra.tex` (the numeric tables of the report) from `output/` and the tables of
notebook 01 (run it first). The remaining tables (`tab_*.tex`) are written by notebook 01.

In [ ]:
import os, sys
from pathlib import Path
from types import SimpleNamespace

# Work from the repository root (the directory that holds the Makefile), wherever the notebook is opened from.
REPO = Path.cwd().resolve()
while not (REPO / "Makefile").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "Makefile").exists(), "open this notebook from inside the repository (the directory with the Makefile)"
os.chdir(REPO)

ROOT      = "output"                 # raw experiment results (the results_* directories of implementation/scripts/submit_all.sh)
DATA_REAL = "output/data_real"       # cell-type labels of the real data set (Tabula Sapiens)
TABLES    = "work/csv"             # summary tables written by summarize.py
SRC       = "work/source"          # source tables of the figures and of the LaTeX tables
TEX       = "work/tex"             # the LaTeX tables of the report
FIGS      = "figures"                # the figures of the report
WORK      = "work/results"           # raw files arranged by report section (intermediate, regenerated by notebook 01)
for d in (TABLES, SRC, TEX, FIGS, WORK):
    os.makedirs(d, exist_ok=True)

## 1. `tables_auto.tex`

### tables_auto.tex

Writes `work/tex/tables_auto.tex` (supplementary tables of the appendix) from the summary tables of notebook 01.

In [ ]:
import csv, sys, collections, statistics as st
cpu, tab = ROOT, TABLES
OUT = f"{TEX}/tables_auto.tex"
rd = lambda p: list(csv.DictReader(open(p)))
o = []
def table(caption, label, header, rows, fmt):
    o.append("\\begin{table}[H]\n\\centering\n\\small\n\\caption{%s}\n\\label{%s}\n\\begin{tabular}{%s}\n\\toprule\n%s \\\\\n\\midrule" % (caption, label, fmt, " & ".join(header)))
    for r in rows: o.append(" & ".join(str(x) for x in r) + " \\\\")
    o.append("\\bottomrule\n\\end{tabular}\n\\end{table}\n")
sp = lambda s: s.split()[0]
# strong scaling, all problems
Ps = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
data = {}
for name, d in (("250k, 400", "E400_big"), ("250k, 512", "E512_big"), ("1M, 768", "E768_1m")):
    data[name] = {(r["variant"], int(r["P"])): r for r in rd(f"{cpu}/results_vcpu_{d}/E_summary.csv")}
rows = []
for P in Ps:
    row = [P]
    for name in data:
        r = data[name].get(("fftdist", P)) or (data[name].get(("fftrep", 1)) if P == 1 else None)
        if r is None: row += ["--", "--"]; continue
        t = float(r["loop_s_median"]) / (60 if "1M" in name else 100)
        row += ["%.3f" % t, sp(r["speedup_vs_rep_P1"]) if P > 1 else "1.00"]
    rows.append(row)
table("Strong scaling with the distributed FFT: time per iteration (s) and speed-up for the three problems (points, boxes). $P=1$ is the serial reference. Speed-up is the median over rounds against the replicated-FFT $P=1$ run of the same round.",
      "tab:app-strong", ["$P$", "t (250k, 400)", "S", "t (250k, 512)", "S", "t (1M, 768)", "S"], rows, "lcccccc")
rows = []
for P in Ps[:7]:
    r = data["250k, 400"].get(("fftrep", P)); 
    if r: rows.append([P, "%.3f" % (float(r["loop_s_median"]) / 100), sp(r["speedup_vs_rep_P1"])])
table("Strong scaling with the replicated FFT (250k points, 400 boxes).", "tab:app-rep", ["$P$", "time per iteration (s)", "speed-up"], rows, "lcc")
# exchange variants
F = rd(f"{cpu}/results_vcpu_F/F_summary.csv"); rows = []
nm = {"F1_allgather_block": "A1 allgather, blocking", "F2_allgather_nonblock": "A2 allgather, non-blocking", "F3_ghost_block": "A3 ghost, blocking", "F4_ghost_nonblock": "A4 ghost, non-blocking", "F4poll_ghost_nonblock": "A4 ghost, non-blocking, polling"}
FR = rd(f"{cpu}/results_vcpu_F_rep/F_summary.csv")      # P=128 with 5 repeats (separate job `exch_F_rep`); replaces the 3-repeat rows
for P in (16, 64, 128):
    for r in (FR if P == 128 else F):
        if int(r["P"]) == P and r["variant"] in nm and r["data"] == "g250k":
            rows.append([P, nm[r["variant"]], "%.1f" % float(r["iter_ms"]), "%.1f" % float(r["exch_ms"]), "%.1f" % (100 * float(r["exch_share"])), "%.1f" % (100 * float(r["iter_spread"]))])
table("Coordinate-exchange variants (\\mbox{Q1-exchange}), 250k points, 3 repeats ($P=128$: 5 repeats, separate job). Exchange = pack, communication, wait and unpack time per iteration; spread = (max-min)/mean of the iteration time over the repeats.",
      "tab:app-exch", ["$P$", "variant", "iteration (ms)", "exchange (ms)", "share (\\%)", "spread (\\%)"], rows, "llcccc")
# placement
lay = lambda r: "72/72/72/40" if (int(r["P"]), int(r["ranks_per_node"])) == (256, 72) else r["ranks_per_node"]
rows = [[r["P"], r["nodes"], lay(r), "%.0f" % (1000 * float(r["sec_per_iter"])), "%.0f" % (1000 * float(r["fft"])), "%.0f" % (1000 * float(r["grid_comm"])),
         "%.0f" % (1000 * float(r["fft_transpose"])), "%.0f" % (1000 * float(r["pot_allgather"]))] for r in rd(f"{tab}/place_summary.csv")]
table("Placement, 250k points, 400 boxes, median of 2--5 rounds (ms per iteration).", "tab:app-place",
      ["$P$", "nodes", "ranks/node", "iteration", "FFT", "Reduce\\_scatter", "transposes", "Allgatherv"], rows, "lccccccc")
# weak
rows = [[{"fixed_B1": "fixed grid", "sqrtgrid_fftdist": "grid growing with $N$"}[r["variant"]], r["P"], r["n_rounds"], "%.4f" % float(r["sec_per_iter_mean"]), r["weak_efficiency"]] for r in rd(f"{tab}/weak256_summary.csv")]
table("Weak scaling (\\mbox{Q2-weak}), 5000 points per rank: time per iteration (s) and efficiency $T(1)/T(P)$.", "tab:app-weak", ["variant", "$P$", "rounds", "time (s)", "efficiency"], rows, "lcccc")
# real-data cpu scaling
R = rd(f"{tab}/real_scaling.csv"); t1 = [float(r["sec_per_iter"]) for r in R if r["job"] == "a" and r["P"] == "1"][0]; rows = []
for r in R:
    t = float(r["sec_per_iter"]); P = int(r["P"]); rows.append([("single node" if r["job"] == "a" else "2--4 nodes"), P, r["n_reps"], "%.3f" % t, "%.1f" % (t1 / t), "%.2f" % (t1 / t / P)])
table("Real data on the CPU (\\mbox{Q3-real}): ts50k, 540 boxes, distributed FFT. $T(1)$ comes from the single-node job.", "tab:app-real", ["job", "$P$", "repeats", "s/iter", "speed-up", "efficiency"], rows, "lcccrr")
# official
rows = [[r["data"], {"official": "official", "ours_serial": "ours (serial)", "ours_mpi": "ours (MPI)"}.get(r["impl"], r["impl"]), r["threads_or_P"], r["n"], r["loop_s_mean"], "%s--%s" % (r["loop_s_min"], r["loop_s_max"]), r["speedup_vs_own_1core"], r["trust"], r["kl_common"]]
        for r in rd(f"{tab}/official_compare_summary.csv")]
table("Official FIt-SNE against ours (\\mbox{Q3-official}): time of 1000 iterations (mean and range over three seeds), speed-up against the same code on one core, trustworthiness and exact KL.",
      "tab:app-official", ["data", "code", "cores", "n", "time (s)", "range", "speed-up", "trust.", "KL"], rows, "llcccccrr")
import os; os.makedirs(os.path.dirname(OUT) or ".", exist_ok=True)
open(OUT, "w").write("\n".join(o))
print("wrote", OUT, len(o))

## 2. `tables_extra.tex`

### tables_extra.tex

The tables of the report that are not made by the generators of notebook 01 and the section above: every number is computed here from the result files.


Writes one LaTeX file with these tables (the report's own `tables_extra.tex` and `s4_results.tex` contain the same tabulars with the prose around them):
  tab:strong      speed-up and efficiency, 250k points, 400 boxes, distributed FFT              <- results_vcpu_E400_big/E_summary.csv
  tab:app-scan    serial grid scan (N=100k) and point scan (200 boxes)                           <- results_vanda_cpu_v2/serial/{grid,npts}.csv
  tab:stagetimes  stage times of the strong scaling (Figure R3)                                  <- <tables>/fig2_stage_breakdown.csv
  tab:strong200   strong scaling with 200 boxes                                                  <- results_vcpu_E200/E_summary.csv, results_vcpu_E200_big/E_summary.csv
  tab:official    official FIt-SNE against ours at 1 and 72 cores                                <- <tables>/official_compare_summary.csv
  tab:weakstages  stages of the growing-grid weak-scaling runs                                   <- work/source/weak_stages.csv
  tab:app-realq   quality on Tabula Sapiens against the subset size                              <- real_quality.json (`realquality()` of notebook 01)
  tab:app-realpar / -realsize / -realtypes / -realpairs   parallel against single run; purity by size of a type; best and worst types; most frequent neighbour pairs   <- real_quality.json

In [ ]:
import argparse, csv, json, os, statistics as st

In [ ]:
A = SimpleNamespace(root=ROOT, tables=TABLES, weak_stages=f"{SRC}/weak_stages.csv", real_quality="output/real_quality.json", out=f"{TEX}/tables_extra.tex")
rd = lambda p: list(csv.DictReader(open(p)))
sp = lambda s: float(str(s).split()[0])
O = []

In [ ]:
def table(label, caption, fmt, body, pos="H", extra=""):
    O.append("\\begin{table}[%s]\n\\centering\n\\small\n\\caption{%s}\n\\label{%s}\n%s\\begin{tabular}{%s}\n\\toprule" % (pos, caption, label, extra, fmt))
    O.extend(body)
    O.append("\\bottomrule\n\\end{tabular}\n\\end{table}\n")

In [ ]:
thou = lambda n: "{:,}".format(int(n)).replace(",", "\\,")

In [ ]:
# ---- tab:strong -------------------------------------------------------------------------------------------------------------------
E = rd(f"{A.root}/results_vcpu_E400_big/E_summary.csv")
d = {int(r["P"]): r for r in E if r["variant"] == "fftdist"}
t1 = [float(r["loop_s_median"]) for r in E if r["variant"] == "fftrep" and r["P"] == "1"][0] / 100
Ps = [2, 4, 8, 16, 32, 64, 128, 256, 512]
table("tab:strong", "Speed-up and efficiency, 250k points, 400 boxes, \\emph{distributed FFT} ($T(1)=%.2f$\\,s per iteration in the paired job). The tables for the other two problems are in Appendix~\\ref{app:tables}." % t1,
      "l" + "c" * len(Ps),
      ["$P$ & " + " & ".join(map(str, Ps)) + " \\\\\n\\midrule",
       "Speed-up & " + " & ".join("%.2f" % sp(d[P]["speedup_vs_rep_P1"]) for P in Ps) + " \\\\",
       "Efficiency & " + " & ".join("%.2f" % (sp(d[P]["speedup_vs_rep_P1"]) / P) for P in Ps) + " \\\\"])

In [ ]:
# ---- tab:app-scan -----------------------------------------------------------------------------------------------------------------
S = f"{A.root}/results_vanda_cpu_v2/serial"
g = {}
for r in rd(f"{S}/grid.csv"):
    g.setdefault(int(r["boxes"]), []).append(r)
g[200] = [r for r in rd(f"{S}/npts.csv") if int(r["N"]) == 100000]     # the 200-box point of the scan is the 100k point of the point scan (same configuration)
med = lambda v, c: st.median(float(x[c]) for x in v)          # median over the repetitions of one configuration
rows = ["\\multicolumn{4}{l}{(a) grid scan, $N=100$k}\\\\\nboxes & nodes per dimension & FFT convolution (s) & iteration (s) \\\\\n\\midrule"]
for b in sorted(g):
    rows.append("%d & %d & %.3f & %.3f \\\\" % (b, int(g[b][0]["grid_dim"]), med(g[b], "med_fft"), med(g[b], "med_total")))
body_a = rows
n = {}
for r in rd(f"{S}/npts.csv"):
    n.setdefault(int(r["N"]), []).append(r)
rows = ["\\multicolumn{5}{l}{(b) point scan, 200 boxes}\\\\\n$N$ & FFT convolution (s) & FFT of the kernel (s) & rest (s) & iteration (s) \\\\\n\\midrule"]
for N in sorted(n):
    fft, ks, tot = med(n[N], "med_fft"), med(n[N], "med_kernel_spec"), med(n[N], "med_total")
    rows.append("%s & %.3f & %.3f & %.3f & %.3f \\\\" % (thou(N), fft, ks, tot - fft - ks, tot))
O.append("\\begin{table}[H]\n\\centering\n\\small\n\\caption{Serial iteration (one core, 90 neighbors). (a) Fixed $N=100$k points, growing grid. (b) Fixed grid of 200 boxes (600 nodes per dimension), growing $N$; the rest is the iteration without the two FFT terms (attraction, spread, gather, update).}\n\\label{tab:app-scan}\n\\begin{tabular}{rrrr}\n\\toprule")
O.extend(body_a)
O.append("\\bottomrule\n\\end{tabular}\n\n\\medskip\n\\begin{tabular}{rrrrr}\n\\toprule")
O.extend(rows)
O.append("\\bottomrule\n\\end{tabular}\n\\end{table}\n")

In [ ]:
# ---- tab:stagetimes ---------------------------------------------------------------------------------------------------------------
T = rd(f"{A.tables}/fig2_stage_breakdown.csv")
f0 = lambda x: "%.0f" % float(x)
def row(name, f): return name + " & " + " & ".join(f0(f(r)) for r in T) + " \\\\"
table("tab:stagetimes", "Numbers behind Figure~\\ref{fig:stages}. The iteration time also contains the stages not listed and waiting, and the stages overlap.", "lcccccc",
      ["Stage (ms per iteration) & " + " & ".join(r["P"] for r in T) + " \\\\\n\\midrule",
       row("FFT compute", lambda r: r["steady_fft"]), row("Grid \\emph{Reduce\\_scatter}", lambda r: r["steady_grid_comm"]),
       row("Two FFT transposes", lambda r: r["steady_fft_transpose"]), row("Potential \\emph{Allgatherv}", lambda r: r["steady_pot_allgather"]),
       row("Sum of the four stages", lambda r: sum(round(float(r[k])) for k in ("steady_fft", "steady_grid_comm", "steady_fft_transpose", "steady_pot_allgather"))),   # sum of the rounded entries
       row("Whole iteration", lambda r: r["steady_iter"])])

In [ ]:
# ---- tab:strong200 ----------------------------------------------------------------------------------------------------------------
a = {int(r["P"]): r for r in rd(f"{A.root}/results_vcpu_E200/E_summary.csv") if r["variant"] == "fftdist"}
b = {int(r["P"]): r for r in rd(f"{A.root}/results_vcpu_E200_big/E_summary.csv") if r["variant"] == "fftdist"}
P2 = [2, 4, 8, 16, 32, 64, 128, 256]
src = lambda P: (b[P], "$^\\dagger$") if P in b and P > 64 else (a[P], "")
table("tab:strong200", "Strong scaling with the \\emph{distributed FFT}, 250k points, 200 boxes (median of three rounds; $^\\dagger$ second job).", "l" + "c" * len(P2),
      ["$P$ & " + " & ".join(map(str, P2)) + " \\\\\n\\midrule",
       "Time per iteration (s) & " + " & ".join("%.3f%s" % (float(src(P)[0]["loop_s_median"]) / 100, src(P)[1]) for P in P2) + " \\\\",
       "Speed-up & " + " & ".join("%.2f%s" % (sp(src(P)[0]["speedup_vs_rep_P1"]), src(P)[1]) for P in P2) + " \\\\",
       "Efficiency & " + " & ".join("%.2f%s" % (float(src(P)[0]["efficiency"]), src(P)[1]) for P in P2) + " \\\\"])

In [ ]:
# ---- tab:official -----------------------------------------------------------------------------------------------------------------
Ofc = rd(f"{A.tables}/official_compare_summary.csv")
get = lambda data, impl, c: float([r for r in Ofc if r["data"] == data and r["impl"] == impl and int(r["threads_or_P"]) == c][0]["loop_s_mean"])
rows = [" & \\multicolumn{2}{c}{1 core (s)} & \\multicolumn{2}{c}{72 cores (s)} & \\multicolumn{2}{c}{Speed-up 72 vs.\\ 1} \\\\\nData & official & ours & official & ours & official & ours \\\\\n\\midrule"]
for dname in ("g50k", "ts50k"):
    o1, s1, o72, m72 = get(dname, "official", 1), get(dname, "ours_serial", 1), get(dname, "official", 72), get(dname, "ours_mpi", 72)
    rows.append("%s & %.1f & %.1f & %.1f & %.1f & %.1f & %.1f \\\\" % (dname, o1, s1, o72, m72, o1 / o72, s1 / m72))
table("tab:official", "Loop time of the official code (72 threads) and of ours (serial; MPI $P=72$); means of three seeds.", "lcccccc", rows)

In [ ]:
# ---- tab:weakstages ---------------------------------------------------------------------------------------------------------------
W = rd(A.weak_stages)
rows = ["$P$ & iteration & FFT & \\emph{Reduce\\_scatter} & transposes & \\emph{Allgatherv} & grid stages & share \\\\\n\\midrule"]
for r in W:
    rows.append("%s & %s & %s & %s & %s & %s & %s & %.0f\\%% \\\\" % (r["P"], r["iter_ms"], r["fft_ms"], r["reduce_scatter_ms"], r["transposes_ms"], r["allgatherv_ms"], r["grid_stages_ms"], 100 * float(r["grid_share"])))
table("tab:weakstages", "Stages of the growing-grid weak-scaling runs (ms per iteration, slowest rank, steady state, two repeats). The grid stages are the sum of the \\emph{Reduce\\_scatter}, the transposes and the \\emph{Allgatherv}, and the share is their part of the iteration.",
      "rrrrrrrr", rows, pos="!htb", extra="\\setlength{\\tabcolsep}{4pt}\n")

In [ ]:
# ---- tab:app-realq ----------------------------------------------------------------------------------------------------------------
if os.path.exists(A.real_quality):
    q = json.load(open(A.real_quality)); res = q["results"]
    full = {int(k): v[0] for k, v in res["ts"]["P1_r1"]["per_type"].items()}                  # cells per type in the full set
    rare = {k for k, c in full.items() if c < 1000}
    rows = ["Data set & cells & types & trustworthiness & kNN recall & label purity & purity of rare types \\\\\n\\midrule"]
    for key, nm in (("ts10k", "ts10k"), ("ts50k", "ts50k"), ("ts200k", "ts200k"), ("ts", "ts (all)")):
        runs = [v for k, v in res[key].items() if k.startswith("P1_")]
        pt = runs[0]["per_type"]; cells = sum(v[0] for v in pt.values())
        rp = st.mean(v[1] for k, v in pt.items() if int(k) in rare and v[0] >= 5)
        rows.append("%s & %s & %d & %.4f & %.3f & %.3f & %.3f \\\\" % (nm, thou(cells), len(pt), st.mean(r["trustworthiness"] for r in runs), st.mean(r["knn_recall"] for r in runs),
                                                                      st.mean(r["purity_all"] for r in runs), rp))
    table("tab:app-realq", "Quality on Tabula Sapiens against the size of the subset (one GPU, fixed grid of 540 boxes, 1000 iterations). The rare types are those with fewer than 1000 cells in the full set; their purity is the mean over the types present in the subset (at least 5 cells).",
          "lrrcccc", rows)

    # ---- the four tables on the cell types of the full data set (mean over the three P=1 repeats) --------------------------------
    names = q["names"]; R = res["ts"]
    p1 = [k for k in R if k.startswith("P1_")]
    cells = {int(t): v[0] for t, v in R[p1[0]]["per_type"].items()}
    pur = {t: st.mean(R[k]["per_type"][str(t)][1] for k in p1) for t in cells}
    short = lambda n: (n.replace("naive thymus-derived CD4-positive, alpha-beta T cell", "naive CD4 T cell").replace("CD4-positive, alpha-beta memory T cell", "CD4 memory T cell")
                       .replace("effector CD4-positive, alpha-beta T cell", "effector CD4 T cell"))
    rep_spread = max(max(R[k]["per_type"][str(t)][1] for k in p1) - min(R[k]["per_type"][str(t)][1] for k in p1) for t in cells)
    def change(keys):
        m = {t: st.mean(R[k]["per_type"][str(t)][1] for k in keys) for t in cells}
        return max(abs(m[t] - pur[t]) for t in cells)
    def avg(keys, f): return st.mean(R[k][f] for k in keys)
    one = [k for k in R if k.startswith("P2s_")]; two = [k for k in R if k.startswith("P2x_")]
    rng = lambda keys, f: sorted({"%.3f" % R[k][f] for k in keys})
    knn_p2 = rng(one + two, "knn_recall"); knn_p2 = knn_p2[0] if len(knn_p2) == 1 else "%s--%s" % (knn_p2[0], knn_p2[-1])
    table("tab:app-realpar", "Parallel against single run on Tabula Sapiens (all 483k cells, GPU, fixed grid of 540 boxes). The last column is the largest change of the purity of a single type: for $P=2$ against $P=1$ (means of three repeats), and for $P=1$ the spread between the three repeats.",
          "lcccc", ["Run & trustworthiness & kNN recall & label purity & largest change of a type's purity \\\\\n\\midrule",
                    "$P=1$ & %.4f & %.3f & %.3f & %.3f (between repeats) \\\\" % (avg(p1, "trustworthiness"), avg(p1, "knn_recall"), avg(p1, "purity_all"), rep_spread),
                    "$P=2$, one node & %.4f & %s & %.3f & %.3f (against $P=1$) \\\\" % (avg(one, "trustworthiness"), "%.3f--%.3f" % (min(R[k]["knn_recall"] for k in one), max(R[k]["knn_recall"] for k in one)), avg(one, "purity_all"), change(one)),
                    "$P=2$, two nodes & %.4f & %.3f & %.3f & %.3f (against $P=1$) \\\\" % (avg(two, "trustworthiness"), avg(two, "knn_recall"), avg(two, "purity_all"), change(two))])
    bins = ((0, 100, "fewer than 100"), (100, 1000, "100--1000"), (1000, 10000, "1000--10\\,000"), (10000, 10 ** 9, "more than 10\\,000"))
    rows = ["Cells of a type & types & weighted purity \\\\\n\\midrule"]
    for lo, hi, lab in bins:
        ts = [t for t in cells if lo <= cells[t] < hi]
        rows.append("%s & %d & %.3f \\\\" % (lab, len(ts), sum(cells[t] * pur[t] for t in ts) / sum(cells[t] for t in ts)))
    table("tab:app-realsize", "Weighted purity of the annotated types by the number of cells of a type (all 483k cells, %d types)." % len(cells), "lrr", rows)
    big = sorted((t for t in cells if cells[t] >= 1000), key=lambda t: -pur[t]); small = sorted(big, key=lambda t: pur[t])
    rows = ["Group & type & purity \\\\\n\\midrule"]
    rows += ["best & %s & %.3f \\\\" % (short(names[t]), pur[t]) for t in big[:5]] + ["worst & %s & %.3f \\\\" % (short(names[t]), pur[t]) for t in small[:5]]
    table("tab:app-realtypes", "The best and the worst separated types (all 483k cells). Best: types with at least 1000 cells.", "llr", rows)
    # the neighbour pairs of the stored list (the 15 most frequent) that the report shows: ranks 1-6 and the fibroblast / mesenchymal stem cell pair
    # (the pairs between effector T cell and NK T cell subtypes are left out); display names as in the report
    SHOW = ((0, "CD4-positive and CD8-positive $\\alpha\\beta$ T cell"), (1, "B cell and memory B cell"), (2, "T cell and effector CD4 T cell"), (3, "memory B cell and naive B cell"),
            (4, "memory CD4 T cell and naive CD4 T cell"), (5, "prostate basal cell and prostate epithelial cell"), (11, "fibroblast and mesenchymal stem cell"))
    conf = R["P1_r1"]["confused"]
    rows = ["Pair of types & links \\\\\n\\midrule"] + ["%s & %dk \\\\" % (txt, round(conf[i][2] / 1000)) for i, txt in SHOW]
    table("tab:app-realpairs", "The most frequent neighbor pairs of different types in the embedding (all 483k cells), with the number of neighbor links.", "lr", rows)
os.makedirs(os.path.dirname(A.out) or ".", exist_ok=True)
open(A.out, "w").write("\n".join(O))
print("wrote", A.out)

## 3. One csv per table of the report

Converts every LaTeX table written by notebooks 01 and 03 (`work/tex/`) into a csv file `tables/<label>.csv` with the same rows and columns as in the report, and writes `tables/INDEX.csv` (label, file, caption).

In [ ]:
import csv, glob, re

def tex_text(s):
    """LaTeX cell text -> plain text"""
    s = re.sub(r"~?\\ref\{([^}]*)\}", r" \1", s)   # a cross reference is kept as its label
    s = re.sub(r"\\(?:emph|textbf|textit|mbox|texttt)\{([^{}]*)\}", r"\1", s)
    for a, b in ((r"^\dagger", "†"), (r"\dagger", "†"), (r"\Delta", "Delta"), (r"\ ", " "), (r"\,", ""), (r"\%", "%"), (r"\_", "_"), (r"\times", "x"), (r"\alpha", "alpha"), (r"\beta", "beta"), (r"\le", "<="), (r"\approx", "~"),
                 ("--", "-"), ("~", " "), ("$", ""), ("{", ""), ("}", "")):
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip()

def tabular_rows(tab):
    """rows of one LaTeX tabular as lists of cells (multicolumn cells are padded with empty cells)"""
    body = tab.split("\\toprule", 1)[1].rsplit("\\bottomrule", 1)[0]
    rows = []
    for line in re.split(r"\\\\\s*\n|\\\\$", body):
        line = re.sub(r"\\(?:midrule|cmidrule(?:\([^)]*\))?\{[^}]*\})", "", line).strip()
        if not line:
            continue
        cells = []
        for c in line.split("&"):
            m = re.match(r"\s*\\multicolumn\{(\d+)\}\{[^}]*\}\{(.*)\}\s*$", c, re.S)
            if m:
                cells += [tex_text(m.group(2))] + [""] * (int(m.group(1)) - 1)
            else:
                cells.append(tex_text(c))
        rows.append(cells)
    return rows

# one csv per table of the report (file name = LaTeX label), plus INDEX.csv (label, file, caption)
os.makedirs("tables", exist_ok=True)
for f in glob.glob("tables/*.csv"):
    os.remove(f)
index = []
for f in sorted(glob.glob(f"{TEX}/*.tex")):
    txt = open(f).read()
    for m in re.finditer(r"\\begin\{table\}.*?\\end\{table\}", txt, re.S):
        t = m.group(0)
        label = re.search(r"\\label\{([^}]*)\}", t).group(1)
        caption = tex_text(re.search(r"\\caption\{(.*)\}\n", t).group(1))
        rows = tabular_rows(re.search(r"\\begin\{tabular\}.*?\\end\{tabular\}", t, re.S).group(0))
        name = label.replace("tab:", "tab_") + ".csv"
        with open(f"tables/{name}", "w", newline="") as o:
            csv.writer(o, lineterminator="\n").writerows(rows)
        index.append([label, name, caption])
with open("tables/INDEX.csv", "w", newline="") as o:
    w = csv.writer(o, lineterminator="\n"); w.writerow(["label", "file", "caption"]); w.writerows(sorted(index))
print("wrote", len(index), "tables to tables/ (index: tables/INDEX.csv)")
